Gold Layer: This layer represeants the final stage of the data, it would implement star schema to have the ability to figure out the main KPIS

First creating the gold database

In [0]:
%sql
create database if not exists cash_flow_project.cash_flow_gold

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

Dimentions

1. Date Dimention

In [0]:
dim_date = spark.sql("""
    SELECT
        CAST(date_format(date, 'yyyyMMdd') AS INT) AS date_key,
        date,
        year(date)        AS year,
        month(date)       AS month,
        dayofmonth(date)  AS day,
        quarter(date)     AS quarter,
        date_format(date, 'MMMM')  AS month_name,
        date_format(date, 'EEEE')  AS day_name,
        date_format(date, 'yyyy-MM') AS month_key,
        CASE WHEN dayofweek(date) IN (1,7) THEN true ELSE false END AS is_weekend
    FROM (
        SELECT explode(sequence(
            to_date('2022-01-01'),
            to_date('2023-12-31'),
            interval 1 day
        )) AS date
    )
""")

In [0]:
(dim_date.write.format("delta").mode("overwrite")
 .saveAsTable("cash_flow_project.cash_flow_gold.dim_date"))

Account Dim

In [0]:
accounts_data = [
    (1, "Checking Main",      "Checking"),
    (2, "Checking Secondary", "Checking"),
    (3, "Credit Card",        "Credit Card"),
]
dim_accounts = spark.createDataFrame(
    accounts_data, ["account_key", "account_name", "account_type"]
)
(dim_accounts.write.format("delta").mode("overwrite")
 .saveAsTable("cash_flow_project.cash_flow_gold.dim_accounts"))

Categories Dim

In [0]:
categories_data = [
    (101, "Sales Revenue",    "Income"),
    (102, "COGS",             "Expense"),
    (103, "Operating Expense","Expense"),
    (104, "Payroll",          "Expense"),
    (105, "Marketing",        "Expense"),
    (106, "Utilities",        "Expense"),
    (107, "Supplies & COGS",  "Expense"),
    (108, "Internal Transfer","Transfer"),
    (109, "Other",            "Other"),
]
dim_categories = spark.createDataFrame(
    categories_data, ["category_key", "category_name", "category_type"]
)
(dim_categories.write.format("delta").mode("overwrite")
 .saveAsTable("cash_flow_project.cash_flow_gold.dim_categories"))

Employee Dim 

In [0]:
payroll_silver = spark.table("cash_flow_project.cash_flow_silver.gusto_payroll")
dim_employee = (
    payroll_silver
    .select("employee_id", "employee_name", "role", "type", "account")
    .distinct()
    .withColumnRenamed("type", "pay_type")
    .withColumn("employee_key",
        F.dense_rank().over(Window.orderBy("employee_id")))
)
(dim_employee.write.format("delta").mode("overwrite")
 .saveAsTable("cash_flow_project.cash_flow_gold.dim_employee"))

In [0]:
display(dim_employee)

Products DIM

In [0]:
sales_silver = spark.table("cash_flow_project.cash_flow_silver.coffee_shop_sales")
dim_product = (
    sales_silver
    .select("product_id", "product_category", "product_type", "product_detail", "unit_price")
    .groupBy("product_id", "product_category", "product_type", "product_detail")
    .agg(F.avg("unit_price").cast("decimal(8,2)").alias("standard_unit_price"))
    .orderBy("product_id")
)
(dim_product.write.format("delta").mode("overwrite")
 .saveAsTable("cash_flow_project.cash_flow_gold.dim_product"))

Fact Tables

In [0]:
main_silver = spark.table("cash_flow_project.cash_flow_silver.checking_accounts_main")
sec_silver  = spark.table("cash_flow_project.cash_flow_silver.checking_account_secondary")
cc_silver   = spark.table("cash_flow_project.cash_flow_silver.credit_card_account")
dim_date_df = spark.table("cash_flow_project.cash_flow_gold.dim_date")
dim_acc     = spark.table("cash_flow_project.cash_flow_gold.dim_accounts")
dim_cat     = spark.table("cash_flow_project.cash_flow_gold.dim_categories")


Fact table for cash transaction

In [0]:
main_txns = (
    main_silver
    .withColumn("account_name", F.lit("Checking Main"))
    .select(
        "transaction_id", "date", "description", "category",
        "type", "amount", "signed_amount", "balance", "account_name"
    )
)

In [0]:
sec_txns = (
    sec_silver
    .withColumn("account_name", F.lit("Checking Secondary"))
    .select(
        "transaction_id", "date", "description", "category",
        "type", "amount", "signed_amount", "balance", "account_name"
    )
)

In [0]:
cc_txns = (
    cc_silver
    .withColumn("category",
        F.when(F.col("vendor") == "Facebook Ads",    "Marketing")
         .when(F.col("vendor") == "Local Print Shop","Marketing")
         .when(F.col("vendor") == "Utility Company", "Utilities")
         .when(F.col("vendor") == "Coffee Supplier", "Supplies & COGS")
         .when(F.col("vendor") == "Payment to CC",   "Internal Transfer")
         .otherwise("Other"))
    .withColumn("account_name", F.lit("Credit Card"))
    .withColumnRenamed("vendor", "description")
    .select(
        "transaction_id", "date", "description", "category",
        "type", "amount", "signed_amount", "balance", "account_name"
    )
)

In [0]:
all_txns = main_txns.union(sec_txns).union(cc_txns)
fact_cash = (
    all_txns
    .join(dim_date_df.select("date", "date_key"), on="date", how="left")
    .join(dim_acc.select("account_name", "account_key"), on="account_name", how="left")
    .join(
        dim_cat.select("category_name", "category_key"),
        all_txns.category == dim_cat.category_name,
        how="left"
    )
    .select(
        "transaction_id",
        "date_key",
        "date",
        "account_key",
        "account_name",
        F.coalesce(F.col("category_key"), F.lit(109)).alias("category_key"),
        F.col("category").alias("category_name"),
        "type",
        "description",
        F.round("amount", 2).alias("amount"),
        F.round("signed_amount", 2).alias("signed_amount"),
        F.round("balance", 2).alias("balance"),
    )
)

In [0]:
(fact_cash.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("cash_flow_project.cash_flow_gold.fact_cash_transactions"))

Fact Table fo shortage detection

In [0]:
monthly_window = Window.orderBy("month_key").rowsBetween(-3, -1)

In [0]:
monthly_checking = (
    main_silver
    .withColumn("month_key", F.date_format("date", "yyyy-MM"))
    .groupBy("month_key")
    .agg(
        F.sum(F.when(F.col("category") == "Sales Revenue",
                     F.col("amount"))).alias("cash_revenue"),
        F.sum(F.when(F.col("category") == "COGS",
                     F.col("amount"))).alias("cogs_expense"),
        F.sum(F.when(F.col("category") == "Operating Expense",
                     F.col("amount"))).alias("operating_expense"),
        F.last("balance").alias("month_end_balance"),
    )
)

In [0]:
monthly_cc = (
    cc_silver
    .filter(F.col("type") == "Debit")
    .withColumn("month_key", F.date_format("date", "yyyy-MM"))
    .withColumn("cc_category",
        F.when(F.col("vendor").isin("Facebook Ads", "Local Print Shop"), "Marketing")
         .when(F.col("vendor") == "Utility Company", "Utilities")
         .when(F.col("vendor") == "Coffee Supplier", "Supplies & COGS")
         .otherwise("Other"))
    .groupBy("month_key")
    .agg(
        F.sum("amount").alias("total_cc_spend"),
        F.sum(F.when(F.col("cc_category") == "Marketing",F.col("amount"))).alias("marketing_spend"),
        F.sum(F.when(F.col("cc_category") == "Utilities",F.col("amount"))).alias("utilities_spend"),
        F.sum(F.when(F.col("cc_category") == "Supplies & COGS",F.col("amount"))).alias("supplies_spend"),
    )
)

In [0]:
monthly_payroll = (
    payroll_silver
    .withColumn("month_key", F.date_format("pay_date", "yyyy-MM"))
    .groupBy("month_key")
    .agg(
        F.sum("amount").alias("total_payroll"),
        F.sum(F.when(F.col("type") == "Employee Pay",   F.col("amount"))).alias("employee_payroll"),
        F.sum(F.when(F.col("type") == "Contractor Pay", F.col("amount"))).alias("contractor_payroll"),
    )
)

In [0]:
monthly_pos = (
    sales_silver
    .withColumn("month_key", F.date_format("transaction_date", "yyyy-MM"))
    .groupBy("month_key")
    .agg(
        F.sum("revenue").alias("pos_gross_revenue"),
        F.countDistinct("transaction_id").alias("num_transactions"),
        F.sum("transaction_qty").alias("total_units_sold"),
        F.sum(F.when(F.col("product_category") == "Coffee",   F.col("revenue"))).alias("coffee_revenue"),
        F.sum(F.when(F.col("product_category") == "Tea",      F.col("revenue"))).alias("tea_revenue"),
        F.sum(F.when(F.col("product_category") == "Bakery",   F.col("revenue"))).alias("bakery_revenue"),
    )
)

In [0]:
display(monthly_pos)

In [0]:
fact_monthly = (
    monthly_checking
    .join(monthly_cc,      on="month_key", how="left")
    .join(monthly_payroll, on="month_key", how="left")
    .join(monthly_pos,     on="month_key", how="left")
    .fillna(0)
    # Total expenses
    .withColumn("total_expenses",
        F.round(
            F.coalesce(F.col("cogs_expense"),       F.lit(0)) +
            F.coalesce(F.col("operating_expense"),  F.lit(0)) +
            F.coalesce(F.col("total_payroll"),      F.lit(0)) +
            F.coalesce(F.col("total_cc_spend"),     F.lit(0))
        , 2))
    # Net cash flow
    .withColumn("net_cash_flow",
        F.round(F.col("cash_revenue") - F.col("total_expenses"), 2))
    # Payroll % of revenue
    .withColumn("payroll_pct_of_revenue",
        F.when(F.col("cash_revenue") > 0,
               F.round(F.col("total_payroll") / F.col("cash_revenue") * 100, 1))
         .otherwise(F.lit(None)))
)


In [0]:
display(fact_monthly)

In [0]:
fact_monthly_with_rolling = (
    fact_monthly
    .withColumn("avg_3m_revenue",
        F.avg("cash_revenue").over(monthly_window).cast("decimal(12,2)"))
    .withColumn("avg_3m_expenses",
        F.avg("total_expenses").over(monthly_window).cast("decimal(12,2)"))
    .withColumn("avg_3m_balance",
        F.avg("month_end_balance").over(monthly_window).cast("decimal(12,2)"))
)

In [0]:
fact_monthly_final = (
    fact_monthly_with_rolling
    .withColumn("shortage_flag",
        F.when(F.col("net_cash_flow") < 0,
               F.lit("CRITICAL"))
        .when(
            F.col("avg_3m_expenses").isNotNull() &
            (F.col("month_end_balance") < F.col("avg_3m_expenses") * 1.5),
            F.lit("WARNING"))
        .when(
            F.col("avg_3m_expenses").isNotNull() &
            (F.col("total_expenses") > F.col("avg_3m_expenses") * 1.2),
            F.lit("EXPENSE_SPIKE"))
        .when(
            F.col("avg_3m_revenue").isNotNull() &
            (F.col("cash_revenue") < F.col("avg_3m_revenue") * 0.8),
            F.lit("INCOME_DROP"))
        .otherwise(F.lit("HEALTHY"))
    )
    .withColumn("shortage_severity",
        F.when(F.col("shortage_flag") == "CRITICAL",      4)
        .when(F.col("shortage_flag") == "EXPENSE_SPIKE",  3)
        .when(F.col("shortage_flag") == "INCOME_DROP",    2)
        .when(F.col("shortage_flag") == "WARNING",        1)
        .otherwise(0))
    .withColumn("needs_ai_suggestion",
        F.when(F.col("shortage_flag") != "HEALTHY", True).otherwise(False))
    .orderBy("month_key")
)

In [0]:
(fact_monthly_final.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("cash_flow_project.cash_flow_gold.fact_monthly_cashflow"))

In [0]:
%sql
select * from cash_flow_project.cash_flow_gold.fact_monthly_cashflow

Fact for sales daily transaction

In [0]:
fact_sales = (
    sales_silver
    .join(
        dim_date_df.select("date", "date_key"),
        sales_silver.transaction_date == dim_date_df.date,
        how="left"
    )
    .join(dim_product.select("product_id", "product_key"
          if "product_key" in dim_product.columns else "product_id"),
          on="product_id", how="left")
    .select(
        "transaction_id",
        "date_key",
        F.col("transaction_date").alias("date"),
        F.date_format("transaction_date", "yyyy-MM").alias("month_key"),
        "product_id",
        "product_category",
        "product_type",
        "product_detail",
        "transaction_qty",
        "unit_price",
        "revenue",
        "hour",
    )
)

In [0]:
(fact_sales.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("cash_flow_project.cash_flow_gold.fact_sales"))

In [0]:
%sql
select * from cash_flow_project.cash_flow_gold.fact_sales

fact_payroll_monthly

In [0]:
fact_payroll = (
    payroll_silver
    .withColumn("month_key", F.date_format("pay_date", "yyyy-MM"))
    .join(dim_employee.select("employee_id", "employee_key"), on="employee_id", how="left")
    .groupBy("month_key", "employee_id", "employee_name", "role", "type")
    .agg(
        F.sum("amount").alias("total_paid"),
        F.count("*").alias("num_payments"),
    )
)
(fact_payroll.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("cash_flow_project.cash_flow_gold.fact_payroll_monthly"))